# Pairwise Feature Engineering — the "main" script

**What changed vs. the version you gave me, and why (most severe first):**

1. **`iterrows()` + `s1.loc[r.entity_id]` / `database.loc[r.candidate_id]` per row is the single biggest
   bottleneck here.** Label-based `.loc` lookups repeated for every candidate pair, at the scale this
   pipeline targets (tens of millions of pairs after Step 5's union), is the kind of thing that turns a
   job that should take minutes into one that takes days. Replaced with one upfront vectorized `merge`
   (brings in every needed column for both sides at once) followed by a plain `zip()` loop over already-
   materialized lists for the parts that genuinely can't be vectorized (rapidfuzz string comparisons, set
   operations).
2. **Schema mismatch with the actual Step 5 output.** This script expects a single `tfidf_score` and a
   `rank` column - my Step 5 rewrite produces `tfidf_word_score` / `tfidf_char_score` (no unified score,
   since collapsing word- and char-level TF-IDF into one number throws away exactly the distinction that
   makes having both worthwhile) and no `rank` column at all. Adjusted to match, and the `rank`/`reciprocal_rank`
   idea is replaced with a *within-entity* rank computed properly at the end (see point 6).
3. **`fuzz.WRatio(...)` was labeled `"name_jaro"`.** `WRatio` is rapidfuzz's blended heuristic ratio, not
   Jaro-Winkler similarity - keeping that name would misdescribe your own feature set in the methodology
   write-up. Renamed to `name_wratio`, and added a real `name_jaro_winkler` feature using
   `rapidfuzz.distance.JaroWinkler`.
4. **`pin_match` hardcoded a 6-digit (India-only) length check** - the exact same open-set-country issue
   flagged earlier in this project. Replaced with `postal_match`, which reuses Step 1's `postal_code` field
   (already country-aware with a generic fallback for France). The old version would have silently never
   fired for US or French records.
5. **`city_match` / `state_match` didn't actually identify a city or a state** - they were just
   "≥2 shared address tokens" / "≥3 shared address tokens" relabeled as if that meant something structural.
   Renamed to `address_token_overlap_ge2` / `address_token_overlap_ge3` so the methodology doc doesn't
   overstate what's being measured.
6. **`score_gap` (`1 - bge_cosine`) and `cosine_percentile` (a literal copy of `bge_cosine`) were both
   redundant with `bge_cosine` itself** - zero new information, pure collinearity. Removed, and replaced
   with what they were presumably meant to approximate: `bge_rank_in_entity`, `gap_to_leader`, and
   `leader_margin`, computed once per S1 entity across its own candidates - the actual "gap between top
   and runner-up" signal the earlier pipeline write-up called for.
7. **`name_prefix_match`/`name_suffix_match` could false-positive on very short/empty names** (e.g. two
   empty strings compare equal on `""[:5] == ""[:5]`) - guarded with a minimum-length check.
8. **NaN handling was inconsistent and, in a couple of spots, silently wrong.** `tfidf_score` was 0-filled
   while `bge_cosine` was left as NaN, with no stated reason; and the `x or default` idiom used for a couple
   of derived values doesn't do what it looks like it does when `x` is NaN (`nan or 0` evaluates to `nan`,
   since NaN is truthy in Python - a classic gotcha). Standardized: the primary similarity scores
   (`bge_cosine`, `tfidf_word_score`, `tfidf_char_score`) are left as NaN when a blocker never found the
   pair - tree models can use that missingness as signal - with explicit `found_by_*` boolean flags so the
   distinction survives even after a later imputation step. Interaction/average features use explicit
   `pd.isna()` checks, not the `or` idiom.
9. **Duplicate `entity_id` in Source 1 or the S2+S3 database would silently multiply rows through the new
   `merge`** (a many-to-many join, not an error) - asserted against this before merging, and added a warning
   count for any pair whose `entity_id`/`candidate_id` didn't match anything in the cleaned data at all.
10. **No skip-if-already-done check and no auto-derived shard name.** Added an early return if the output
    file already exists, and `SHARD_NAME` is now derived from the pair file's own filename rather than a
    separately hand-typed variable - one less place for the path and the label to drift out of sync when
    switching shards.

Code is kept in one cell as asked, wrapped in a `main()` for a clean early-return on the skip case rather
than nesting the entire script under one `if`.

In [ ]:
# ==========================================================
# PATH CONFIGURATION (CHANGE ONLY THIS)
# ==========================================================

TRAIN_SHARD = r"PASTE_TRAIN_SHARD_PATH"
TEST_SHARD  = r"PASTE_TEST_SHARD_PATH"

SOURCE1_CLEANED = r"PASTE_SOURCE1_CLEANED_FOLDER"
SOURCE2_CLEANED = r"PASTE_SOURCE2_CLEANED_FOLDER"
SOURCE3_CLEANED = r"PASTE_SOURCE3_CLEANED_FOLDER"

OUTPUT_FOLDER = r"PASTE_OUTPUT_FOLDER"

MODE = "train"                 # train / test

# ==========================================================
# IMPORTS
# ==========================================================

from pathlib import Path
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from tqdm import tqdm


def main():
    # ------------------------------------------------------
    # RESOLVE PATHS (shard name derived from the pair file itself)
    # ------------------------------------------------------
    pair_file = Path(TRAIN_SHARD if MODE == "train" else TEST_SHARD)
    shard_name = pair_file.stem

    Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    output_file = Path(OUTPUT_FOLDER) / f"{MODE}_features_{shard_name}.parquet"

    if output_file.exists():
        print(f"[SKIP] {output_file} already exists.")
        return

    # ------------------------------------------------------
    # LOAD INPUT
    # ------------------------------------------------------
    print("Loading candidate pairs and cleaned datasets...")
    pairs = pd.read_parquet(pair_file)

    def load_folder(folder):
        files = sorted(Path(folder).glob("part_*_meta.parquet"))
        if not files:
            raise FileNotFoundError(f"No 'part_*_meta.parquet' files found in {folder}.")
        return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

    s1 = load_folder(SOURCE1_CLEANED)
    s2 = load_folder(SOURCE2_CLEANED)
    s3 = load_folder(SOURCE3_CLEANED)
    database = pd.concat([s2, s3], ignore_index=True)

    # Duplicate entity_ids here would silently multiply rows through the
    # merge below (a many-to-many join, not an error) - check first.
    assert not s1["entity_id"].duplicated().any(), "Duplicate entity_id in Source 1 - check Step 1 output."
    assert not database["entity_id"].duplicated().any(), "Duplicate entity_id in S2+S3 - check Step 1 output."

    # ------------------------------------------------------
    # VECTORIZED MERGE (replaces per-row s1.loc[...] / database.loc[...])
    # ------------------------------------------------------
    join_cols = [
        "entity_id", "name_full_clean", "address_clean",
        "name_tokens", "address_tokens", "address_numbers",
        "postal_code", "country",
    ]

    s1_q = s1[join_cols].add_suffix("_q").rename(columns={"entity_id_q": "entity_id"})
    db_c = database[join_cols].add_suffix("_c").rename(columns={"entity_id_c": "candidate_id"})

    merged = pairs.merge(s1_q, on="entity_id", how="left").merge(db_c, on="candidate_id", how="left")

    unmatched = int(merged["name_full_clean_q"].isna().sum() + merged["name_full_clean_c"].isna().sum())
    if unmatched:
        print(f"  WARNING: {unmatched:,} join failures - some entity_id/candidate_id values in the "
              f"pair file weren't found in the cleaned Source 1 / S2+S3 data. Their features will be NaN.")

    # ------------------------------------------------------
    # HELPERS
    # ------------------------------------------------------
    def to_set(x):
        if pd.isna(x) or x == "":
            return set()
        return set(str(x).split("|"))

    def jaccard(a, b):
        u = a | b
        return len(a & b) / len(u) if u else 0.0

    def overlap(a, b):
        return len(a & b) / len(a) if a else 0.0

    def clean_str(x):
        return "" if pd.isna(x) else x

    def safe_num(x):
        # NOT `x or 0.0` - that idiom silently returns NaN unchanged,
        # since NaN is truthy in Python. Explicit isna() check instead.
        return 0.0 if pd.isna(x) else float(x)

    # ------------------------------------------------------
    # FEATURE ENGINEERING
    # ------------------------------------------------------
    needed_cols = [
        "entity_id", "candidate_id",
        "bge_cosine", "tfidf_word_score", "tfidf_char_score",
        "name_full_clean_q", "name_full_clean_c",
        "address_clean_q", "address_clean_c",
        "name_tokens_q", "name_tokens_c",
        "address_tokens_q", "address_tokens_c",
        "address_numbers_q", "address_numbers_c",
        "postal_code_q", "postal_code_c",
        "country_q", "country_c",
    ]
    cols = merged[needed_cols]

    rows = []
    for (entity_id, candidate_id, bge_cosine, tfidf_word, tfidf_char,
         nq, nc, aq, ac, ntq_raw, ntc_raw, atq_raw, atc_raw,
         numq_raw, numc_raw, pinq, pinc, cq, cc) in tqdm(
            zip(*[cols[c] for c in needed_cols]), total=len(cols)):

        nq, nc = clean_str(nq), clean_str(nc)
        aq, ac = clean_str(aq), clean_str(ac)
        pinq, pinc = clean_str(pinq), clean_str(pinc)
        cq, cc = clean_str(cq), clean_str(cc)

        ntq, ntc = to_set(ntq_raw), to_set(ntc_raw)
        atq, atc = to_set(atq_raw), to_set(atc_raw)
        numq, numc = to_set(numq_raw), to_set(numc_raw)

        name_wratio = fuzz.WRatio(nq, nc) / 100
        name_jaro_winkler = JaroWinkler.normalized_similarity(nq, nc)
        name_lev = fuzz.ratio(nq, nc) / 100
        addr_lev = fuzz.ratio(aq, ac) / 100
        addr_jac = jaccard(atq, atc)

        bge_val = safe_num(bge_cosine)
        tfidf_word_val = safe_num(tfidf_word)
        tfidf_char_val = safe_num(tfidf_char)

        sim_values = [v for v in (bge_cosine, tfidf_word, tfidf_char, name_wratio, addr_jac) if not pd.isna(v)]

        rows.append({
            "entity_id": entity_id,
            "candidate_id": candidate_id,

            # Semantic / lexical retrieval scores - left as NaN (not
            # 0-filled) when a blocker never found this pair; NaN and
            # "found it with a genuinely low score" are different
            # information. found_by_* keeps that distinction explicit
            # even after any later imputation collapses the NaN.
            "bge_cosine": bge_cosine,
            "tfidf_word_score": tfidf_word,
            "tfidf_char_score": tfidf_char,
            "found_by_bge": int(not pd.isna(bge_cosine)),
            "found_by_tfidf_word": int(not pd.isna(tfidf_word)),
            "found_by_tfidf_char": int(not pd.isna(tfidf_char)),

            # Name
            "name_wratio": name_wratio,              # rapidfuzz WRatio - a blended heuristic, NOT Jaro
            "name_jaro_winkler": name_jaro_winkler,   # actual Jaro-Winkler similarity
            "name_levenshtein": name_lev,
            "name_jaccard": jaccard(ntq, ntc),
            "name_token_overlap": overlap(ntq, ntc),
            "name_word_count": len(ntq),
            "candidate_name_word_count": len(ntc),
            "name_count_diff": abs(len(ntq) - len(ntc)),
            "name_prefix_match": int(len(nq) >= 5 and len(nc) >= 5 and nq[:5] == nc[:5]),
            "name_suffix_match": int(len(nq) >= 5 and len(nc) >= 5 and nq[-5:] == nc[-5:]),
            "common_name_tokens": len(ntq & ntc),

            # Address
            "address_jaccard": addr_jac,
            "address_levenshtein": addr_lev,
            "address_token_overlap": overlap(atq, atc),
            "address_word_count": len(atq),
            "candidate_address_word_count": len(atc),
            "address_count_diff": abs(len(atq) - len(atc)),
            "number_overlap": overlap(numq, numc),
            "building_number_match": int(len(numq & numc) > 0),
            # Country-agnostic postal match - reuses Step 1's open-set
            # postal_code field instead of a hardcoded 6-digit (India-
            # only) length check, which would never fire for US/France.
            "postal_match": int(bool(pinq) and pinq == pinc),
            # Crude token-overlap-count proxies - NOT verified city/state
            # matches, renamed from "city_match"/"state_match" since
            # nothing here identifies which token is a city or a state.
            "address_token_overlap_ge2": int(len(atq & atc) >= 2),
            "address_token_overlap_ge3": int(len(atq & atc) >= 3),
            "common_address_tokens": len(atq & atc),

            # Country
            "country_match": int(bool(cq) and cq == cc),

            # Numeric
            "numeric_jaccard": jaccard(numq, numc),
            "total_numbers": len(numq),
            "candidate_total_numbers": len(numc),
            "is_source3": int(str(candidate_id).startswith("S3")),

            # Interaction / aggregate - 0-filled here deliberately (an
            # interaction term multiplying a missing signal should read
            # as "no interaction evidence", unlike an average, which
            # should skip missing values rather than being dragged down
            # by them - see avg/max below).
            "semantic_address_interaction": bge_val * addr_jac,
            "semantic_tfidf_interaction": bge_val * max(tfidf_word_val, tfidf_char_val),
            "avg_similarity": (sum(sim_values) / len(sim_values)) if sim_values else 0.0,
            "max_similarity": max(sim_values) if sim_values else 0.0,
        })

    feature_df = pd.DataFrame(rows)

    # ------------------------------------------------------
    # ENTITY-LEVEL RANK / GAP FEATURES
    # ------------------------------------------------------
    # Computed once per S1 entity across its own candidates - replaces
    # the old "score_gap" (just 1-bge_cosine, redundant with bge_cosine
    # itself) and "cosine_percentile" (a literal copy of bge_cosine) with
    # the actual within-entity signal those names implied but didn't
    # compute.
    feature_df["combined_score"] = feature_df[["bge_cosine", "tfidf_word_score", "tfidf_char_score"]].max(axis=1)

    feature_df["bge_rank_in_entity"] = (
        feature_df.groupby("entity_id")["bge_cosine"].rank(ascending=False, method="min")
    )
    leader_score = feature_df.groupby("entity_id")["combined_score"].transform("max")
    feature_df["gap_to_leader"] = leader_score - feature_df["combined_score"]

    second_best = (
        feature_df.groupby("entity_id")["combined_score"]
        .transform(lambda s: s.nlargest(2).min() if len(s) > 1 else s.max())
    )
    feature_df["leader_margin"] = leader_score - second_best  # same value across all rows of one entity

    # ------------------------------------------------------
    # SAVE
    # ------------------------------------------------------
    feature_df.to_parquet(output_file, compression="zstd", index=False)

    print(f"Saved : {output_file}")
    print(f"Rows  : {len(feature_df):,}")


main()